# 🏋️ Google GISLR 250-Class Model Training Pipeline

This notebook trains our advanced sign language recognition models (`Multi-Stream Transformer (MST)` and `ECA Conv-Transformer (ECA-CT)`) on the fully preprocessed Google GISLR dataset containing 250 classes.

### Highlights:
- **Memory-Safe Data Generator**: Utilizes `tf.data.Dataset.from_generator` to dynamically load preprocessed `.npy` shards directly from Google Drive during training without exhausting RAM.
- **Advanced Models**: 250-class variants of our WLASL-50 winning architectures.
- **LR Schedule & Regularization**: Cosine Warmup, Label Smoothing (0.1), and AdamW.

In [1]:
import os
import math
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, GlobalAveragePooling1D, Conv1D, ZeroPadding1D, DepthwiseConv1D, BatchNormalization, MultiHeadAttention, Add

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/My Drive/Final Year Project/Dataset/preprocessed_kaggle"
NUM_CLASSES = 250

Mounted at /content/drive


## 1. Memory-Safe Shard Data Generator

In [2]:
def shard_generator(data_dir, shard_indices):
    """Yields individual samples dynamically from numpy shards."""
    for idx in shard_indices:
        x_path = os.path.join(data_dir, f"X_shard_{idx}.npy")
        y_path = os.path.join(data_dir, f"y_shard_{idx}.npy")

        if not os.path.exists(x_path):
            continue

        # Load single shard (~250MB) into memory
        X = np.load(x_path)
        y_labels = np.load(y_path)

        # Yield samples one-by-one
        for i in range(len(X)):
            # Convert integer label to One-Hot Encoding
            y_ohe = np.zeros(NUM_CLASSES, dtype=np.float32)
            y_ohe[y_labels[i]] = 1.0
            yield X[i], y_ohe

def create_dataset(data_dir, shard_indices, batch_size=32, is_training=True):
    """Creates a high-performance tf.data.Dataset pipeline."""
    dataset = tf.data.Dataset.from_generator(
        lambda: shard_generator(data_dir, shard_indices),
        output_signature=(
            tf.TensorSpec(shape=(30, 71, 6), dtype=tf.float32),
            tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32)
        )
    )

    if is_training:
        dataset = dataset.shuffle(buffer_size=10000) # Shuffles across roughly 2 shards

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Define indices for 90/10 Train-Val split (approx 19 total shards)
train_shards = list(range(0, 17))
val_shards = list(range(17, 19))

train_dataset = create_dataset(DATA_DIR, train_shards, batch_size=64, is_training=True)
val_dataset = create_dataset(DATA_DIR, val_shards, batch_size=64, is_training=False)

print("Datasets successfully initialized!")

Datasets successfully initialized!


## 2. Custom Model Layers

In [3]:
class LandmarkEmbedding(tf.keras.layers.Layer):
    def __init__(self, units, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.units = units

    def build(self, input_shape):
        P = input_shape[2]
        C = input_shape[3]
        flat_dim = P * C
        self.dense1 = tf.keras.layers.Dense(self.units, activation='gelu', kernel_initializer='glorot_uniform')
        self.dense2 = tf.keras.layers.Dense(self.units, kernel_initializer='he_uniform')
        self.dense1.build((None, input_shape[1], flat_dim))
        self.dense2.build((None, input_shape[1], self.units))
        super().build(input_shape)

    def call(self, x):
        P = x.shape[2]
        C = x.shape[3]
        x_flat = tf.reshape(x, (-1, x.shape[1], P * C))

        dense_out = self.dense2(self.dense1(x_flat))
        empty_emb = tf.zeros_like(dense_out)

        is_empty = tf.reduce_sum(tf.abs(x_flat), axis=-1, keepdims=True) == 0.0
        return tf.where(is_empty, empty_emb, dense_out)

class SoftmaxWeightedFusion(tf.keras.layers.Layer):
    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)

    def build(self, input_shape):
        num_inputs = len(input_shape)
        self.landmark_weights = self.add_weight(
            name='landmark_weights',
            shape=(num_inputs,),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        x = tf.stack(inputs, axis=-1)
        weights = tf.nn.softmax(self.landmark_weights)
        x = x * weights
        return tf.reduce_sum(x, axis=-1)

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, units, sequence_length, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.units = units
        self.sequence_length = sequence_length

    def build(self, input_shape):
        self.pos_emb = self.add_weight(
            name='pos_emb',
            shape=(self.sequence_length, self.units),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        return x + self.pos_emb

def transformer_encoder_block(inputs, head_size, num_heads, ff_dim, dropout=0.2):
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(inputs)
    attention_output = tf.keras.layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(x, x)
    x = tf.keras.layers.Add()([inputs, attention_output])

    y = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    ff_output = tf.keras.layers.Dense(ff_dim, activation='gelu')(y)
    ff_output = tf.keras.layers.Dropout(dropout)(ff_output)
    ff_output = tf.keras.layers.Dense(inputs.shape[-1])(ff_output)
    return tf.keras.layers.Add()([x, ff_output])

class ECA(tf.keras.layers.Layer):
    def __init__(self, kernel_size=5, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = tf.keras.layers.Conv1D(1, kernel_size=kernel_size, strides=1, padding="same", use_bias=False)

    def call(self, inputs):
        nn = tf.keras.layers.GlobalAveragePooling1D()(inputs)
        nn = tf.expand_dims(nn, -1)
        nn = self.conv(nn)
        nn = tf.squeeze(nn, -1)
        nn = tf.nn.sigmoid(nn)
        nn = nn[:, None, :]
        return inputs * nn

def Conv1DBlock(channel_size, kernel_size, drop_rate=0.2, expand_ratio=2):
    def apply(inputs):
        channels_in = inputs.shape[-1]
        channels_expand = channels_in * expand_ratio

        x = tf.keras.layers.Dense(channels_expand, activation='swish')(inputs)

        padding_size = kernel_size - 1
        padded = tf.keras.layers.ZeroPadding1D(padding=(padding_size, 0))(x)
        x = tf.keras.layers.DepthwiseConv1D(kernel_size, strides=1, padding='valid', use_bias=False)(padded)

        x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
        x = ECA()(x)

        x = tf.keras.layers.Dense(channel_size)(x)

        if drop_rate > 0:
            x = tf.keras.layers.Dropout(drop_rate)(x)

        if channels_in == channel_size:
            x = tf.keras.layers.add([x, inputs])
        return x
    return apply

def TransformerBlock(dim, num_heads=4, expand=2, drop_rate=0.2):
    def apply(inputs):
        x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(inputs)
        x = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads, dropout=drop_rate)(x, x)
        x = tf.keras.layers.Dropout(drop_rate)(x)
        x = tf.keras.layers.Add()([inputs, x])

        y = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
        y = tf.keras.layers.Dense(dim * expand, activation='swish')(y)
        y = tf.keras.layers.Dense(dim)(y)
        y = tf.keras.layers.Dropout(drop_rate)(y)
        return tf.keras.layers.Add()([x, y])
    return apply

## 3. Model Architectures

In [4]:
def build_mst(input_shape, num_classes):
    inputs = tf.keras.layers.Input(shape=input_shape)

    lips = inputs[:, :, 0:40, :]
    hand = inputs[:, :, 40:61, :]
    pose = inputs[:, :, 61:71, :]

    units = 128
    lips_emb = LandmarkEmbedding(units, name='lips_emb')(lips)
    hand_emb = LandmarkEmbedding(units, name='hand_emb')(hand)
    pose_emb = LandmarkEmbedding(units, name='pose_emb')(pose)

    x = SoftmaxWeightedFusion()([lips_emb, hand_emb, pose_emb])
    x = PositionalEmbedding(units, sequence_length=30)(x)

    x = transformer_encoder_block(x, head_size=32, num_heads=4, ff_dim=256, dropout=0.2)
    x = transformer_encoder_block(x, head_size=32, num_heads=4, ff_dim=256, dropout=0.2)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="mst")

    try:
        import tensorflow_addons as tfa
        optimizer = tfa.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5)
    except Exception:
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )
    return model

def build_eca_ct(input_shape, num_classes, dim=128):
    inputs = tf.keras.layers.Input(shape=input_shape)

    lips = inputs[:, :, 0:40, :]
    hand = inputs[:, :, 40:61, :]
    pose = inputs[:, :, 61:71, :]

    lips_emb = LandmarkEmbedding(dim, name='lips_emb_1st')(lips)
    hand_emb = LandmarkEmbedding(dim, name='hand_emb_1st')(hand)
    pose_emb = LandmarkEmbedding(dim, name='pose_emb_1st')(pose)

    x = SoftmaxWeightedFusion(name='fusion_1st')([lips_emb, hand_emb, pose_emb])
    x = PositionalEmbedding(dim, sequence_length=30, name='pos_emb_1st')(x)

    ksize = 17
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = TransformerBlock(dim, num_heads=4, expand=2, drop_rate=0.2)(x)

    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = TransformerBlock(dim, num_heads=4, expand=2, drop_rate=0.2)(x)

    x = tf.keras.layers.Dense(dim * 2, name='top_conv')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="eca_ct")

    try:
        import tensorflow_addons as tfa
        optimizer = tfa.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5)
    except Exception:
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )
    return model

model_adv = build_mst((30, 71, 6), NUM_CLASSES)
model_eca = build_eca_ct((30, 71, 6), NUM_CLASSES)

model_adv.summary()
model_eca.summary()

Model: "mst"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30, 71, 6) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 30, 40, 6) │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 30, 21, 6) │          0 │ input_layer[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 30, 10, 6) │          0 │ input_layer[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lips_emb            │ (None, 30, 128)   │     47,360 │ get_item[0][0]    │
│ (LandmarkEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_emb            │ (None, 30, 128)   │     32,768 │ get_item_1[0][0]  │
│ (LandmarkEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pose_emb            │ (None, 30, 128)   │     24,320 │ get_item_2[0][0]  │
│ (LandmarkEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_weighted_f… │ (None, 30, 128)   │          3 │ lips_emb[0][0],   │
│ (SoftmaxWeightedFu… │                   │            │ hand_emb[0][0],   │
│                     │                   │            │ pose_emb[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 30, 128)   │      3,840 │ softmax_weighted… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 30, 128)   │        256 │ positional_embed… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 128)   │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 30, 128)   │          0 │ positional_embed… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 30, 256)   │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 30, 256)   │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 30, 128)   │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 30, 128)   │          0 │ add[0][0],        │
│                     │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 128)   │        256 │ add_1[0][0]     

 Total params: 422,013 (1.61 MB)

 Trainable params: 422,013 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

Model: "eca_ct"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 30, 71, 6) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 30, 40, 6) │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 30, 21, 6) │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 30, 10, 6) │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lips_emb_1st        │ (None, 30, 128)   │     47,360 │ get_item_3[0][0]  │
│ (LandmarkEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_emb_1st        │ (None, 30, 128)   │     32,768 │ get_item_4[0][0]  │
│ (LandmarkEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pose_emb_1st        │ (None, 30, 128)   │     24,320 │ get_item_5[0][0]  │
│ (LandmarkEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_1st          │ (None, 30, 128)   │          3 │ lips_emb_1st[0][… │
│ (SoftmaxWeightedFu… │                   │            │ hand_emb_1st[0][… │
│                     │                   │            │ pose_emb_1st[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_emb_1st         │ (None, 30, 128)   │      3,840 │ fusion_1st[0][0]  │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 30, 256)   │     33,024 │ pos_emb_1st[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding1d      │ (None, 46, 256)   │          0 │ dense_18[0][0]    │
│ (ZeroPadding1D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d    │ (None, 30, 256)   │      4,352 │ zero_padding1d[0… │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 256)   │        512 │ depthwise_conv1d… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ eca (ECA)           │ (None, 30, 256)   │          5 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 30, 128)   │     32,896 │ eca[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 30, 128)   │          0 │ dense_19[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 30, 128)   │          0 │ dropout_5[0][0],  │
│                     │                   │            │ pos_emb_1st[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 30, 256)   │     33,024 │ add_4[0][0]     

 Total params: 753,681 (2.88 MB)

 Trainable params: 753,681 (2.88 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Callbacks & Training Loop

In [5]:
class WeightDecayCallback(tf.keras.callbacks.Callback):
    def __init__(self, wd_ratio=0.05):
        super().__init__()
        self.wd_ratio = wd_ratio

    def on_epoch_begin(self, epoch, logs=None):
        if hasattr(self.model.optimizer, 'weight_decay') and hasattr(self.model.optimizer, 'learning_rate'):
            lr = self.model.optimizer.learning_rate
            lr_val = lr.numpy() if hasattr(lr, 'numpy') else float(lr)
            self.model.optimizer.weight_decay = lr_val * self.wd_ratio

def lrfn(current_step, num_warmup_steps=5, lr_max=1e-3, num_cycles=0.50, num_training_steps=50):
    if current_step < num_warmup_steps:
        return lr_max * 2 ** -(num_warmup_steps - current_step)
    else:
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))) * lr_max

LR_SCHEDULE = [lrfn(step, num_warmup_steps=5, lr_max=1e-3, num_training_steps=50) for step in range(50)]
lr_callback = tf.keras.callbacks.LearningRateScheduler(lambda step: LR_SCHEDULE[step], verbose=1)

early_stopping_adv = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

callbacks_list = [early_stopping_adv, lr_callback, WeightDecayCallback()]

In [6]:
print("\n================ Training Multi-Stream Transformer (MST) ================")
history_adv = model_adv.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks_list
)

model_adv.save("/content/drive/My Drive/Final Year Project/models/mst_250.h5")


================ Training Multi-Stream Transformer (MST) ================

Epoch 1: LearningRateScheduler setting learning rate to 3.125e-05.
Epoch 1/50
   1329/Unknown 169s 99ms/step - accuracy: 0.0055 - loss: 5.5295

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1329/1329 ━━━━━━━━━━━━━━━━━━━━ 186s 112ms/step - accuracy: 0.0076 - loss: 5.4943 - val_accuracy: 0.0160 - val_loss: 5.3506 - learning_rate: 3.1250e-05

Epoch 2: LearningRateScheduler setting learning rate to 6.25e-05.
Epoch 2/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 60s 39ms/step - accuracy: 0.0578 - loss: 4.8594 - val_accuracy: 0.1562 - val_loss: 4.1887 - learning_rate: 6.2500e-05

Epoch 3: LearningRateScheduler setting learning rate to 0.000125.
Epoch 3/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 60s 40ms/step - accuracy: 0.1662 - loss: 4.0470 - val_accuracy: 0.3084 - val_loss: 3.4737 - learning_rate: 1.2500e-04

Epoch 4: LearningRateScheduler setting learning rate to 0.00025.
Epoch 4/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 60s 41ms/step - accuracy: 0.2949 - loss: 3.4663 - val_accuracy: 0.4183 - val_loss: 3.0036 - learning_rate: 2.5000e-04

Epoch 5: LearningRateScheduler setting learning rate to 0.0005.
Epoch 5/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 60s 41ms/step - accuracy: 0.3935 - loss: 3.0922 - val_accuracy

In [7]:
print("\n================ Training ECA Conv-Transformer (ECA-CT) ================")
history_eca = model_eca.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks_list
)

model_eca.save("/content/drive/My Drive/Final Year Project/models/eca_ct_250.h5")


================ Training ECA Conv-Transformer (ECA-CT) ================

Epoch 1: LearningRateScheduler setting learning rate to 3.125e-05.
Epoch 1/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 130s 65ms/step - accuracy: 0.0099 - loss: 5.4758 - val_accuracy: 0.0275 - val_loss: 5.1622 - learning_rate: 3.1250e-05

Epoch 2: LearningRateScheduler setting learning rate to 6.25e-05.
Epoch 2/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 68s 46ms/step - accuracy: 0.0609 - loss: 4.8366 - val_accuracy: 0.1728 - val_loss: 4.1698 - learning_rate: 6.2500e-05

Epoch 3: LearningRateScheduler setting learning rate to 0.000125.
Epoch 3/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 67s 45ms/step - accuracy: 0.1773 - loss: 4.0407 - val_accuracy: 0.3367 - val_loss: 3.3650 - learning_rate: 1.2500e-04

Epoch 4: LearningRateScheduler setting learning rate to 0.00025.
Epoch 4/50
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 68s 46ms/step - accuracy: 0.3227 - loss: 3.3774 - val_accuracy: 0.4655 - val_loss: 2.8633 - learning_rate: 2.5000e-04

Epoch 5: Learnin

In [8]:
import numpy as np
import time
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

print("=========================================================")
print("EVALUATING MST FINAL MODEL ON VALIDATION DATASET")
print("=========================================================")

# Extract true labels and predictions
y_true = []
y_pred = []
latencies = []

print("Running validation predictions and profiling latency...")
# Loop over validation dataset to measure latency per sample
for x_batch, y_batch in val_dataset.take(5):  # Sample batches for latency profiling
    batch_true = np.argmax(y_batch.numpy(), axis=-1)
    y_true.extend(batch_true)

    for i in range(x_batch.shape[0]):
        sample = np.expand_dims(x_batch.numpy()[i], axis=0)
        t_start = time.perf_counter()
        pred = model_adv.predict(sample, verbose=0)
        t_end = time.perf_counter()
        latencies.append((t_end - t_start) * 1000)
        y_pred.append(np.argmax(pred[0]))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Compute accuracy
accuracy = accuracy_score(y_true, y_pred)

# Compute precision & recall
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

# Compute latency statistics
avg_latency = np.mean(latencies)
std_latency = np.std(latencies)

print(f"\nAccuracy: {accuracy * 100:.2f}%")
print(f"Precision (Macro): {precision_macro * 100:.2f}%")
print(f"Precision (Weighted): {precision_weighted * 100:.2f}%")
print(f"Recall (Macro): {recall_macro * 100:.2f}%")
print(f"Recall (Weighted): {recall_weighted * 100:.2f}%")
print(f"Average CPU Inference Latency: {avg_latency:.2f} ms (Std: {std_latency:.2f} ms)")

# Confusion Matrix analysis
cm = confusion_matrix(y_true, y_pred)
confusions = []
for r in range(cm.shape[0]):
    for c in range(cm.shape[0]):
        if r != c and cm[r, c] > 0:
            confusions.append((r, c, cm[r, c]))
confusions.sort(key=lambda x: x[2], reverse=True)

# Load gloss index map
try:
    import json
    with open("sign_to_prediction_index_map.json", 'r') as f:
        idx_map = json.load(f)
        gloss_map = {int(v): k for k, v in idx_map.items()}
except Exception:
    gloss_map = {}

print("\nTop 5 Most Confused Pairs:")
for idx, (t_idx, p_idx, count) in enumerate(confusions[:5]):
    t_word = gloss_map.get(t_idx, f"Class {t_idx}")
    p_word = gloss_map.get(p_idx, f"Class {p_idx}")
    print(f"{idx+1}. True: '{t_word}' predicted as '{p_word}' (Count: {int(count)})")


EVALUATING MST FINAL MODEL ON VALIDATION DATASET
Running validation predictions and profiling latency...

Accuracy: 76.25%
Precision (Macro): 72.28%
Precision (Weighted): 81.19%
Recall (Macro): 71.72%
Recall (Weighted): 76.25%
Average CPU Inference Latency: 86.26 ms (Std: 120.87 ms)

Top 5 Most Confused Pairs:
1. True: 'Class 68' predicted as 'Class 66' (Count: 2)
2. True: 'Class 1' predicted as 'Class 106' (Count: 1)
3. True: 'Class 8' predicted as 'Class 174' (Count: 1)
4. True: 'Class 9' predicted as 'Class 166' (Count: 1)
5. True: 'Class 24' predicted as 'Class 185' (Count: 1)


In [9]:
print("=========================================================")
print("EVALUATING ECA CONV-TRANSFORMER (ECA-CT) ON VALIDATION DATASET")
print("=========================================================")

y_true_eca = []
y_pred_eca = []

for x_batch, y_batch in val_dataset.take(5):
    batch_true = np.argmax(y_batch.numpy(), axis=-1)
    y_true_eca.extend(batch_true)

    for i in range(x_batch.shape[0]):
        sample = np.expand_dims(x_batch.numpy()[i], axis=0)
        pred = model_eca.predict(sample, verbose=0)
        y_pred_eca.append(np.argmax(pred[0]))

y_true_eca = np.array(y_true_eca)
y_pred_eca = np.array(y_pred_eca)

accuracy_eca = accuracy_score(y_true_eca, y_pred_eca)
precision_macro_eca, recall_macro_eca, _, _ = precision_recall_fscore_support(y_true_eca, y_pred_eca, average='macro', zero_division=0)

print(f"\nECA-CT Model Accuracy: {accuracy_eca * 100:.2f}%")
print(f"ECA-CT Model Precision (Macro): {precision_macro_eca * 100:.2f}%")
print(f"ECA-CT Model Recall (Macro): {recall_macro_eca * 100:.2f}%")


EVALUATING ECA CONV-TRANSFORMER (ECA-CT) ON VALIDATION DATASET

ECA-CT Model Accuracy: 3.44%
ECA-CT Model Precision (Macro): 1.37%
ECA-CT Model Recall (Macro): 3.32%
